# Customer Churn Prediction

## Feature Engineering

### Objective

Building on the findings from the exploratory data analysis, this notebook transforms the cleaned dataset into a model-ready feature set. This includes encoding categorical variables,engineering new features suggested by the EDA (e.g. the non-linear relationship between products_number and churn), and preparing both a tree-model-friendly and a scaled/linear-model-friendly version of the numeric features.

**Recap of EDA findings guiding this notebook:**
- age,active_member, and balance show the strongest relationships with churn.
- products_number has a non-monotonic (U-shaped) relationship with churn so best treated as categorical rather than raw numeric.
- country and gender show meaningful group differences and should be encoded.
- credit_score, tenure, and estimated_salary show weak individual relationships but are retained in case they contribute in combination with other features.


In [14]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)


In [15]:
df = pd.read_csv("../data/processed/bank_customer_churn_clean.csv")
df.head()


,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   credit_score      10000 non-null  int64  
 1   country           10000 non-null  object 
 2   gender            10000 non-null  object 
 3   age               10000 non-null  int64  
 4   tenure            10000 non-null  int64  
 5   balance           10000 non-null  float64
 6   products_number   10000 non-null  int64  
 7   credit_card       10000 non-null  int64  
 8   active_member     10000 non-null  int64  
 9   estimated_salary  10000 non-null  float64
 10  churn             10000 non-null  int64  
dtypes: float64(2), int64(7), object(2)
memory usage: 859.5+ KB


### Encode gender

gender is binary, so a simple 0/1 mapping is sufficient (no need for one-hot encoding, which would create a redundant column).

In [17]:
df["gender"] = df["gender"].map({"Male": 0, "Female": 1})
df[["gender"]].head()


,gender
0,1
1,1
2,1
3,1
4,1


### Encode country

country is nominal (no inherent order) with three categories, so it is one-hot encoded.
`drop_first=True` avoids the dummy variable trap for linear models; tree-based models are unaffected either way.

In [18]:
df = pd.get_dummies(df, columns=["country"], prefix="country", drop_first=True)
df.head()


,credit_score,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn,country_Germany,country_Spain
0,619,1,42,2,0.00,1,1,1,101348.88,1,0,0
1,608,1,41,1,83807.86,1,0,1,112542.58,0,0,1
2,502,1,42,8,159660.80,3,1,0,113931.57,1,0,0
3,699,1,39,1,0.00,2,0,0,93826.63,0,0,0
4,850,1,43,2,125510.82,1,1,1,79084.10,0,0,1


### Encode products_number

The EDA showed a **non-monotonic** relationship between number of products and churn (2 products = lowest churn, 3-4 products = very high churn). Treating this as a raw numeric feature would force linear models to assume a straight-line relationship that doesn't exist. One-hot encoding lets every model type learn the actual pattern.

In [19]:
# Keep a copy of the original numeric count before encoding, for later use
products_numeric = df["products_number"].astype(int).copy()

df["products_number"] = df["products_number"].astype(int)
df = pd.get_dummies(df, columns=["products_number"], prefix="products", drop_first=True)
df.head()


,credit_score,gender,age,tenure,balance,credit_card,active_member,estimated_salary,churn,country_Germany,country_Spain,products_2,products_3,products_4
0,619,1,42,2,0.00,1,1,101348.88,1,0,0,0,0,0
1,608,1,41,1,83807.86,0,1,112542.58,0,0,1,0,0,0
2,502,1,42,8,159660.80,1,0,113931.57,1,0,0,0,1,0
3,699,1,39,1,0.00,0,0,93826.63,0,0,0,1,0,0
4,850,1,43,2,125510.82,1,1,79084.10,0,0,1,0,0,0


### Engineer new features

Based on patterns observed in the EDA:

- **is_zero_balance** — flag for customers with a balance of exactly 0 (~36% of customers), since the EDA showed retained customers were disproportionately likely to have a zero balance.
- **balance_to_salary_ratio** — balance relative to income, a normalized view of financial standing that raw balance alone doesn't capture.
- **age_group** — binned age, to let models pick up threshold effects (e.g. a jump in churn risk after a certain age) that a continuous variable might smooth over.
- **tenure_by_age** — tenure relative to age, capturing how much of a customer's adult life has been spent with the bank.
- **products_active_interaction** — interaction between active_member and product count, since an inactive member with only 1 product may be a very different risk profile than an active member with 1 product.

In [20]:
# Zero balance flag
df["is_zero_balance"] = (df["balance"] == 0).astype(int)

# Balance to salary ratio (avoid divide-by-zero)
df["balance_to_salary_ratio"] = df["balance"] / df["estimated_salary"].replace(0, np.nan)
df["balance_to_salary_ratio"] = df["balance_to_salary_ratio"].fillna(0)

# Age group buckets, based on the age distribution seen in the EDA
df["age_group"] = pd.cut(
    df["age"],
    bins=[17, 30, 45, 60, 100],
    labels=["18-30", "31-45", "46-60", "60+"]
)
df = pd.get_dummies(df, columns=["age_group"], prefix="age_group", drop_first=True)

# Tenure relative to age
df["tenure_by_age"] = df["tenure"] / df["age"]

# Interaction: active membership x product count (using the original numeric count)
df["products_active_interaction"] = products_numeric * df["active_member"]

df.head()


,credit_score,gender,age,tenure,balance,credit_card,active_member,estimated_salary,churn,country_Germany,country_Spain,products_2,products_3,products_4,is_zero_balance,balance_to_salary_ratio,age_group_31-45,age_group_46-60,age_group_60+,tenure_by_age,products_active_interaction
0,619,1,42,2,0.00,1,1,101348.88,1,0,0,0,0,0,1,0.000000,1,0,0,0.047619,1
1,608,1,41,1,83807.86,0,1,112542.58,0,0,1,0,0,0,0,0.744677,1,0,0,0.024390,1
2,502,1,42,8,159660.80,1,0,113931.57,1,0,0,0,1,0,0,1.401375,1,0,0,0.190476,0
3,699,1,39,1,0.00,0,0,93826.63,0,0,0,1,0,0,1,0.000000,1,0,0,0.025641,0
4,850,1,43,2,125510.82,1,1,79084.10,0,0,1,0,0,0,0,1.587055,1,0,0,0.046512,1


### Separate features and target

Moving churn to the end (or splitting it out entirely) keeps the feature matrix clean for the modeling notebook.

In [21]:
target = df["churn"]
features = df.drop(columns=["churn"])

print("Feature matrix shape:", features.shape)
print("Target shape:", target.shape)


Feature matrix shape: (10000, 20)
Target shape: (10000,)


### Scaled version of numeric features

Tree-based models (Random Forest, XGBoost) don't require scaling, but linear models, SVMs, and
KNN do. Rather than guess which model wins in the next notebook, we save **both** an unscaled
and a scaled version of the feature set, so `05_model_training.ipynb` can pick whichever it
needs without repeating this work.

In [22]:
numeric_cols = [
    "credit_score", "age", "tenure", "balance", "estimated_salary",
    "balance_to_salary_ratio", "tenure_by_age", "products_active_interaction"
]

scaler = StandardScaler()
features_scaled = features.copy()
features_scaled[numeric_cols] = scaler.fit_transform(features[numeric_cols])

features_scaled.head()


,credit_score,gender,age,tenure,balance,credit_card,active_member,estimated_salary,country_Germany,country_Spain,products_2,products_3,products_4,is_zero_balance,balance_to_salary_ratio,age_group_31-45,age_group_46-60,age_group_60+,tenure_by_age,products_active_interaction
0,-0.326221,1,0.293517,-1.041760,-1.225848,1,1,0.021886,0,0,0,0,0,1,-0.035804,1,0,0,-1.009118,0.240195
1,-0.440036,1,0.198164,-1.387538,0.117350,0,1,0.216534,0,1,0,0,0,0,-0.028930,1,0,0,-1.268654,0.240195
2,-1.536794,1,0.293517,1.032908,1.333053,1,0,0.240687,0,0,0,1,0,0,-0.022868,1,0,0,0.587029,-0.909064
3,0.501521,1,0.007457,-1.387538,-1.225848,0,0,-0.108918,0,0,1,0,0,1,-0.035804,1,0,0,-1.254679,-0.909064
4,2.063884,1,0.388871,-1.041760,0.785728,1,1,-0.365276,0,1,0,0,0,0,-0.021154,1,0,0,-1.021491,0.240195


### Quick correlation check on the new engineered features

In [23]:
check_df = features.copy()
check_df["churn"] = target

engineered = ["is_zero_balance", "balance_to_salary_ratio", "tenure_by_age",
              "products_active_interaction"]

check_df[engineered + ["churn"]].corr()["churn"].sort_values(ascending=False)


churn                          1.000000
balance_to_salary_ratio        0.025558
tenure_by_age                 -0.121641
is_zero_balance               -0.122357
products_active_interaction   -0.137902
Name: churn, dtype: float64

### Save engineered datasets

Two CSVs are saved:
- **bank_customer_churn_features.csv** - unscaled, for tree-based models
- **bank_customer_churn_features_scaled.csv** - scaled numeric columns, for linear/distance-based models

Both include the churn target column so `05_model_training.ipynb` can load a single file per approach.

In [24]:
output_unscaled = features.copy()
output_unscaled["churn"] = target
output_unscaled.to_csv("../data/processed/bank_customer_churn_features.csv", index=False)

output_scaled = features_scaled.copy()
output_scaled["churn"] = target
output_scaled.to_csv("../data/processed/bank_customer_churn_features_scaled.csv", index=False)

print("Saved:")
print(" - ../data/processed/bank_customer_churn_features.csv", output_unscaled.shape)
print(" - ../data/processed/bank_customer_churn_features_scaled.csv", output_scaled.shape)


Saved:
 - ../data/processed/bank_customer_churn_features.csv (10000, 21)
 - ../data/processed/bank_customer_churn_features_scaled.csv (10000, 21)


## Summary

**Encoding applied:**
- gender → binary mapped (0/1)
- country → one-hot encoded 
- products_number → one-hot encoded to capture its non-monotonic relationship with churn

**New features engineered:**
- is_zero_balance
- balance_to_salary_ratio
- age_group (binned, one-hot encoded)
- tenure_by_age
- products_active_interaction

**Outputs:**
- bank_customer_churn_features.csv — unscaled feature set (tree-based models)
- bank_customer_churn_features_scaled.csv — scaled feature set (linear/distance-based models)

**Next step:** 
Train and compare candidate models using these feature sets.


In [25]:
pd.crosstab(products_numeric, df["active_member"])

active_member,0,1
products_number,,
1,2521,2563
2,2144,2446
3,153,113
4,31,29
